# logBF vs LOO Diagnostics

Compare per-point local logBF, global logBF proxy, and LOO posterior metrics for dip/jump candidates.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Update this path to the results table you want to inspect.
INPUT_PATH = Path('output/runs/latest/results/lc_events_filtered.parquet')
SAVE_DIR = Path('output/diagnostics/logbf_loo')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

def load_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() in {'.parquet', '.pq'}:
        return pd.read_parquet(path)
    return pd.read_csv(path)

df = load_table(INPUT_PATH)
print(f'Loaded {len(df):,} rows from {INPUT_PATH}')
df.head(3)

In [ ]:
BRANCHES = {
    'dip': {
        'local': 'dip_max_log_bf_local',
        'global': 'dip_bayes_factor',
        'loo': 'dip_max_event_prob',
        'significant': 'dip_significant',
    },
    'jump': {
        'local': 'jump_max_log_bf_local',
        'global': 'jump_bayes_factor',
        'loo': 'jump_max_event_prob',
        'significant': 'jump_significant',
    },
}

for branch, cols in BRANCHES.items():
    missing = [c for c in cols.values() if c not in df.columns]
    if missing:
        raise KeyError(f"{branch}: missing columns {missing}")

def finite(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors='coerce')
    return x[np.isfinite(x)]

def branch_frame(frame: pd.DataFrame, branch: str) -> pd.DataFrame:
    cols = BRANCHES[branch]
    out = pd.DataFrame({
        'local_logbf': pd.to_numeric(frame[cols['local']], errors='coerce'),
        'global_logbf': pd.to_numeric(frame[cols['global']], errors='coerce'),
        'loo_max_prob': pd.to_numeric(frame[cols['loo']], errors='coerce'),
        'significant': frame[cols['significant']].fillna(False).astype(bool),
    })
    return out.replace([np.inf, -np.inf], np.nan).dropna(subset=['local_logbf', 'global_logbf', 'loo_max_prob'])


In [ ]:
for branch in ('dip', 'jump'):
    d = branch_frame(df, branch)
    if d.empty:
        print(f'{branch}: no finite rows')
        continue

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(d['local_logbf'], bins=50, alpha=0.85, color='steelblue')
    axes[0].set_title(f'{branch}: local logBF')

    axes[1].hist(d['global_logbf'], bins=50, alpha=0.85, color='darkorange')
    axes[1].set_title(f'{branch}: global logBF')

    axes[2].hist(d['loo_max_prob'].clip(0, 1), bins=50, alpha=0.85, color='seagreen')
    axes[2].set_title(f'{branch}: LOO max posterior')

    for ax in axes:
        ax.set_xlabel('value')
        ax.set_ylabel('count')

    fig.suptitle(f'{branch.upper()} distributions', y=1.03)
    fig.tight_layout()
    fig.savefig(SAVE_DIR / f'{branch}_distributions.png', dpi=140, bbox_inches='tight')
    plt.show()

In [ ]:
for branch in ('dip', 'jump'):
    d = branch_frame(df, branch)
    if d.empty:
        continue

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    hb = axes[0].hexbin(d['local_logbf'], d['global_logbf'], gridsize=45, mincnt=1, cmap='viridis')
    axes[0].set_xlabel('local logBF')
    axes[0].set_ylabel('global logBF')
    axes[0].set_title(f'{branch}: local vs global')
    fig.colorbar(hb, ax=axes[0], label='count')

    hb = axes[1].hexbin(d['global_logbf'], d['loo_max_prob'], gridsize=45, mincnt=1, cmap='magma')
    axes[1].set_xlabel('global logBF')
    axes[1].set_ylabel('LOO max posterior')
    axes[1].set_title(f'{branch}: global vs LOO')
    fig.colorbar(hb, ax=axes[1], label='count')

    hb = axes[2].hexbin(d['local_logbf'], d['loo_max_prob'], gridsize=45, mincnt=1, cmap='plasma')
    axes[2].set_xlabel('local logBF')
    axes[2].set_ylabel('LOO max posterior')
    axes[2].set_title(f'{branch}: local vs LOO')
    fig.colorbar(hb, ax=axes[2], label='count')

    fig.tight_layout()
    fig.savefig(SAVE_DIR / f'{branch}_pairwise.png', dpi=140, bbox_inches='tight')
    plt.show()

In [ ]:
def summarize_branch(frame: pd.DataFrame, branch: str) -> pd.DataFrame:
    d = branch_frame(frame, branch)
    if d.empty:
        return pd.DataFrame()

    qs = d[['local_logbf', 'global_logbf', 'loo_max_prob']].quantile([0.01, 0.05, 0.5, 0.95, 0.99]).T
    qs.columns = [f'q{int(q * 100):02d}' for q in qs.columns]

    corr = d[['local_logbf', 'global_logbf', 'loo_max_prob']].corr(method='spearman')
    corr_rows = pd.DataFrame({
        'q01': np.nan, 'q05': np.nan, 'q50': np.nan, 'q95': np.nan, 'q99': np.nan,
    }, index=[
        f'spearman(local,global)={corr.loc["local_logbf","global_logbf"]:.3f}',
        f'spearman(global,loo)={corr.loc["global_logbf","loo_max_prob"]:.3f}',
        f'spearman(local,loo)={corr.loc["local_logbf","loo_max_prob"]:.3f}',
    ])

    return pd.concat([qs, corr_rows])

for branch in ('dip', 'jump'):
    print(f'\n{branch.upper()} summary')
    display(summarize_branch(df, branch))

for branch in ('dip', 'jump'):
    d = branch_frame(df, branch)
    if d.empty:
        continue
    print(f'\n{branch.upper()} by significance flag')
    display(d.groupby('significant')[['local_logbf', 'global_logbf', 'loo_max_prob']].median())